## 두 코드의 결정적인 분기점은 **README의 [Step 3]와 [Step 5] 사이, 즉 '최적 설계안을 도출하는 핵심 알고리즘'**에서 발생합니다.


### 1. 공통 구간 (README Step 1 ~ Step 2)

두 파일 모두 프로젝트의 기초가 되는 데이터 처리 로직은 동일하게 공유합니다.

* **Step 1 (물리 로직):** 단순히 평균값이 아닌, 300초 시계열 중 가장 위험한 지점을 찾는 **'절댓값 Max Peak'** 추출 로직을 공통으로 사용합니다.
* **Step 2 (데이터 증강):** **XGBoost** 대리 모델을 학습시켜 10만 개의 가상 설계 조합에 대한 결과를 초고속으로 예측하는 단계를 동일하게 수행합니다.

---

### 2. 결정적 분기점 (README Step 3 이후)

두 파일은 **'우수한 설계를 정의하고 찾아가는 방식'**에서 완전히 다른 기법을 선택했습니다.

#### **A. `Step 1 XGBoost + Step 2 (4) Step 6+.ipynb` : [역설계 및 시각적 해설 중심]**

이 파일은 README의 **[Step 4] '딥러닝 기반 역설계'** 개념을 핵심으로 삼습니다.

* **핵심 기법:** **1D-CNN 역방향 모델**
* **분기점:** 타겟 곡선을 입력하면 곧바로 설계 변수($P1 \sim P6$)를 출력하는 'AI 모델의 추론 능력'에 집중합니다.
* **특징:** AI가 내놓은 초안(Draft)이 왜 물리적으로 타당한지 **구조적 진화(Structural Evolution)** 과정을 엔지니어에게 설명(XAI)하고 시각화하는 리포트 성격이 강합니다.

#### **B. `Flipchip surrogate v2_clear.ipynb` : [강건 최적화 시스템 중심]**

이 파일은 README의 **[Step 5] '강건 최적화'** 단계를 극대화하여 시스템화한 형태입니다.

* **핵심 기법:** **NSGA-II (유전 알고리즘) + GPR (가우시안 과정 회귀)**
* **분기점:** 단순히 곡선을 따라가는 역설계를 넘어, 수만 번의 진화 연산을 통해 **수학적 최적해(Knee Point)**를 확정하는 데 집중합니다.
* **특징:** 예측 오차(불확실성)를 고려하는 **GPR**을 도입하여, 어떤 상황에서도 제품이 깨지지 않는 가장 안전하고 탄탄한(Robust) 최종 수치를 랭킹(Rank)별로 산출하는 설계 도구 성격이 강합니다.

---

### 3. 요약 비교

| 구분 | `Step 6+` 파일 (역설계 중심) | `v2_clear` 파일 (최적화 중심) |
| --- | --- | --- |
| **README 핵심 단계** | **Step 4 (Inverse Design)** | **Step 5 (Robust Optimization)** |
| **주요 알고리즘** | 1D-CNN (딥러닝 역방향 모델) | NSGA-II (유전 알고리즘) + GPR |
| **선택의 기준** | 타겟 곡선과의 **형상 일치성** | 뒤틀림/박리의 **수학적 최소화** |
| **최종 결과물** | 초안 대비 구조 변화 해설 리포트 | 신뢰성 기반 Rank 1 최종 설계 확정안 |

**결론적으로**, `Step 6+` 파일은 AI가 **"왜 이렇게 설계했는가?"**를 설명하는 데 목적이 있고, `v2_clear` 파일은 **"실제 양산에 적용할 가장 완벽한 수치는 무엇인가?"**를 결정하는 데 목적이 있습니다. 이 지점이 두 파일이 README를 해석하여 코드로 구현할 때 갈라진 가장 큰 분기점입니다.

## **두 파일의 각 단계에서 사용된 구체적인 기법은 다음과 같습니다.**

### 1. [Step 1] 대리 모델 학습 및 데이터 전처리

두 파일 모두 기초 데이터의 물리적 타당성을 확보하는 기법은 동일하게 적용되었습니다.

* **공통 기법: 절댓값 Max Peak 추출 (`abs().idxmax()`)**
* 300초 시계열 데이터 중 단순 최댓값이 아닌, 압축(-)과 인장(+)력을 모두 고려한 가장 파괴적인 피크 지점을 추출하여 학습 데이터의 품질을 높였습니다.


* **공통 기법: XGBoost (Extreme Gradient Boosting)**
* CAE 시뮬레이션을 대신해 10만 개의 가상 설계 조합(P1~P6)에 대한 결과를 초고속으로 예측하는 대리 모델로 사용되었습니다.



---

### 2. [Step 2 ~ 3] 데이터 증강 및 타겟 추출

여기서부터는 데이터의 '필터링'과 '타겟팅' 방식에서 세부적인 차이가 나타납니다.

* **`Step 1 XGBoost + Step 2 (4) Step 6+.ipynb`**
* **기법:** **유토피아 타겟 스케일링 (x0.9)**
* README의 지침대로 파레토 후보군(최근 실행 결과 19개)의 평균 곡선에 0.9를 곱해 딥러닝 모델이 지향할 '도전적 목표치'를 생성했습니다.


* **`Flipchip surrogate v2_clear.ipynb`**
* **기법:** **파레토 최적화 (Pareto Frontier Selection)**
* README의 경고대로 `WarpMax`와 `T_Tip_Peel` 단 2개 지표만을 목적으로 삼아 상위 우수 데이터를 엄격하게 선별하는 알고리즘을 강화했습니다.



---

### 3. [Step 4] 역설계 (Inverse Design) - **주요 분기점**

이 단계에서 두 파일이 사용하는 핵심 AI 알고리즘이 완전히 갈립니다.

* **`Step 1 XGBoost + Step 2 (4) Step 6+.ipynb`**
* **기법:** **1D-CNN (Convolutional Neural Network)**
* 타겟 시계열 곡선을 이미지나 텐서 형태로 입력받아, 이를 구현할 수 있는 설계 수치를 출력하는 **이미지 인식 기반 역방향 모델**을 사용했습니다.


* **`Flipchip surrogate v2_clear.ipynb`**
* 이 파일은 역설계 모델 단독 사용보다는, 다음 단계인 **최적화 알고리즘의 초기값(Initial Guess)**을 생성하는 용도로 AI를 활용합니다.



---

### 4. [Step 5] 최적화 (Optimization) - **최종 결론 방식**

최종 설계안을 확정 짓는 '의사결정' 기법에서 가장 큰 차이가 발생합니다.

* **`Step 1 XGBoost + Step 2 (4) Step 6+.ipynb`**
* **기법:** **Differential Evolution (미분 진화 알고리즘)**
* 역설계 AI가 내놓은 **초안(P1: 0.7639 등)**을 바탕으로 수치를 미세 조정하여 최종안(**P1: 0.0653 등**)으로 진화시키는 방식을 취합니다.


* **`Flipchip surrogate v2_clear.ipynb`**
* **기법:** **NSGA-II (다목적 유전 알고리즘) + GPR (가우시안 과정 회귀)**
* 단순 최적화를 넘어, **GPR**을 통해 AI 예측의 불확실성(오차 범위)까지 계산합니다. 불확실성이 낮은 '안전한 영역'에서 최적의 밸런스 점(**Knee Point**)을 찾아 **Rank 1** 설계안을 확정하는 **강건 최적화(Robust Optimization)** 기법을 사용했습니다.



---

### 💡 요약 비교표

| 단계 | `Step 6+` 파일 (역설계·해설 중심) | `v2_clear` 파일 (최적화·강건성 중심) |
| --- | --- | --- |
| **Step 1** | XGBoost (Max Peak 로직) | XGBoost (Max Peak 로직) |
| **Step 4** | **1D-CNN (시계열 역방향 추론)** | 알고리즘 초기값 생성용 AI |
| **Step 5** | Differential Evolution (수치 보정) | **NSGA-II + GPR (강건 최적화)** |
| **특징** | **구조적 진화 해석** (왜 바뀌었나?) | **신뢰성 랭킹** (어느 것이 가장 안전한가?) |

결론적으로 **`Step 6+`** 파일은 **"AI의 추론과 해설"**에, **`v2_clear`** 파일은 **"수학적 최적화와 결과의 신뢰성"**에 방점을 둔 기법들을 사용했습니다.

## **각 파일이 선택한 알고리즘과 그에 따른 기술적 차이점을 정리해 드립니다.**

두 파일은 **Step 1(XGBoost)**까지는 동일한 궤적을 그리지만, **Step 4와 Step 5**에서 **"딥러닝 기반의 추론"**을 중시하느냐, **"수학적 최적화의 정밀도"**를 중시하느냐에 따라 알고리즘이 확연히 갈립니다.

---

### 1. 주요 사용 알고리즘 비교

| 단계 | **`Step 1 XGBoost + Step 2 (4) Step 6+.ipynb`** | **`Flipchip surrogate v2_clear.ipynb`** |
| --- | --- | --- |
| **Step 1~2** | **XGBoost** (대리 모델링) | **XGBoost** (대리 모델링) |
| **Step 4** | **1D-CNN** (역설계 딥러닝 모델) | **NSGA-II** 연산을 위한 초기 Seed 생성 |
| **Step 5** | **Differential Evolution** (미분 진화 알고리즘) | **NSGA-II** (다목적 유전 알고리즘) + **GPR** |
| **Step 5-2** | - | **Gaussian Process Regression** (불확실성 예측) |

---

### 2. 알고리즘별 상세 역할 및 기술적 차이

#### **① 1D-CNN vs. NSGA-II (추론 vs. 탐색)**

* **1D-CNN (`Step 6+` 파일 사용):** * **역할:** 타겟 시계열 곡선(300초 데이터)을 입력하면, 학습된 패턴을 바탕으로 $P1 \sim P6$ 값을 즉각 **'추론'**합니다.
* **차이:** 사람이 곡선을 보고 "이건 P1이 얇아야겠네"라고 판단하는 과정을 자동화한 것입니다. 속도가 매우 빠르지만, 학습 데이터 범위를 벗어난 '완벽히 새로운 최적점'을 찾는 데는 한계가 있습니다.


* **NSGA-II (`v2_clear` 파일 사용):**
* **역할:** 적자생존의 원리를 이용해 수천 개의 설계 조합을 서로 교배/변이시키며 최적의 조합을 **'탐색'**합니다.
* **차이:** 딥러닝처럼 정답을 맞추는 것이 아니라, 수만 번의 시뮬레이션을 통해 뒤틀림과 박리 응력이 동시에 낮은 '황금 밸런스' 지점을 찾아냅니다.



#### **② Differential Evolution vs. GPR (보정 vs. 강건성)**

* **Differential Evolution (`Step 6+` 파일 사용):**
* **역할:** AI가 내놓은 초안(Draft)의 오차를 줄이기 위해 수치를 미세하게 깎고 다듬는 **보정 알고리즘**입니다.
* **특징:** CNN이 틀릴 수 있는 수치적 디테일을 수학적으로 보완하여 최종안을 확정합니다.


* **GPR (Gaussian Process Regression) (`v2_clear` 파일 사용):**
* **역할:** 최적화된 결과값이 얼마나 믿을만한지 **'확률적 신뢰도'**를 계산합니다.
* **특징:** "이 설계안은 예측 오차가 적은 안전한 구간에 있습니다"라는 정보를 제공하여, 실제 공정에서 발생할 수 있는 오차까지 고려한 **강건 설계(Robust Design)**를 가능하게 합니다.



---

### 3. 핵심 차이 요약: "어떤 결과를 원하는가?"

1. **`Step 1 XGBoost + Step 2 (4) Step 6+.ipynb` (설명 지향형)**
* **핵심 철학:** AI가 타겟 곡선의 특징을 얼마나 잘 이해하고 설계로 변환하는지 **해석**하는 데 집중합니다.
* **알고리즘 강점:** 1D-CNN을 통해 시계열 데이터의 '형태(Shape)'를 설계 변수에 투영하는 능력이 탁월합니다.


2. **`Flipchip surrogate v2_clear.ipynb` (성능 지향형)**
* **핵심 철학:** 물리적 지표(Warp, Peel)를 극한으로 낮추는 **수학적 최적점**을 찾는 데 집중합니다.
* **알고리즘 강점:** NSGA-II와 GPR의 결합으로, 사람이 상상하기 힘든 정밀한 수치 밸런스를 찾아내며 예측의 신뢰도까지 보장합니다.



**결론적으로**, `Step 6+` 파일은 **"AI가 학습한 물리적 직관"**을 확인하고 싶을 때 적합하며, `v2_clear` 파일은 **"실제 공정에 적용할 가장 안전하고 뛰어난 수치"**를 얻고 싶을 때 적합한 알고리즘 구성을 가지고 있습니다.